In [1]:
import os

print(os.listdir("/kaggle/input/ovariancancer-final"))


FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/input/ovariancancer-final'

In [ ]:
# Option 2
BASE_DIR = "/kaggle/input/ovariancancer-final"


In [ ]:
print(os.listdir(BASE_DIR))


['OvarianCancer_Final']


In [ ]:
BASE_DIR = "/kaggle/input/ovariancancer-final/OvarianCancer_Final"

import os
print(os.listdir(BASE_DIR))



['val', 'test', 'train']


In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

TRAIN_PATH = BASE_DIR + "/train"
VAL_PATH   = BASE_DIR + "/val"
TEST_PATH  = BASE_DIR + "/test"

IMG_SIZE = (227, 227)
BATCH_SIZE = 32   # Best for Kaggle GPU

train_datagen = ImageDataGenerator(rescale=1./255)
val_datagen   = ImageDataGenerator(rescale=1./255)
test_datagen  = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    TRAIN_PATH,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=True
)

val_generator = val_datagen.flow_from_directory(
    VAL_PATH,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=True
)

test_generator = test_datagen.flow_from_directory(
    TEST_PATH,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=False
)

print("Training samples :", train_generator.samples)
print("Validation samples:", val_generator.samples)
print("Testing samples  :", test_generator.samples)


2026-02-06 16:05:59.921892: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1770393960.174817      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1770393960.247845      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1770393960.866646      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770393960.866696      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770393960.866699      55 computation_placer.cc:177] computation placer alr

Found 13756 images belonging to 5 classes.
Found 2946 images belonging to 5 classes.
Found 2952 images belonging to 5 classes.
Training samples : 13756
Validation samples: 2946
Testing samples  : 2952


Build VGG16 Model

In [2]:
from tensorflow.keras.applications import VGG16
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
import tensorflow as tf

NUM_CLASSES = train_generator.num_classes

# Load VGG16 backbone
base_model = VGG16(
    weights="imagenet",
    include_top=False,
    input_shape=(227, 227, 3)
)

# Freeze convolution layers
for layer in base_model.layers:
    layer.trainable = False

# ----- Custom Classifier -----
x = base_model.output
x = GlobalAveragePooling2D()(x)

x = Dense(512, activation="elu")(x)
x = Dropout(0.4)(x)

x = Dense(256, activation="elu")(x)
x = Dropout(0.4)(x)

outputs = Dense(NUM_CLASSES, activation="softmax")(x)

vgg16_model = Model(inputs=base_model.input, outputs=outputs)

# Compile model with YOUR parameters
vgg16_model.compile(
    optimizer=Adam(learning_rate=0.0003),
    loss=tf.keras.losses.CategoricalCrossentropy(),
    metrics=["accuracy"]
)

vgg16_model.summary()


NameError: name 'train_generator' is not defined

In [ ]:
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau

checkpoint = ModelCheckpoint(
    "vgg16_best.keras",
    monitor="val_loss",
    save_best_only=True,
    mode="max",
    verbose=1
)

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True
)

lr_scheduler = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.3,
    patience=3,
    min_lr=1e-7,
    verbose=1
)


In [ ]:
history_vgg16 = vgg16_model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=30,
    callbacks=[checkpoint, early_stop, lr_scheduler]
)


/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/30


I0000 00:00:1770394025.018617     158 service.cc:152] XLA service 0x789090013bf0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1770394025.018657     158 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1770394025.018661     158 service.cc:160]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1770394025.669318     158 cuda_dnn.cc:529] Loaded cuDNN version 91002


  1/430 ━━━━━━━━━━━━━━━━━━━━ 2:01:57 17s/step - accuracy: 0.1250 - loss: 1.9340

I0000 00:00:1770394039.972913     158 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


430/430 ━━━━━━━━━━━━━━━━━━━━ 0s 238ms/step - accuracy: 0.4708 - loss: 1.2945
Epoch 1: val_loss improved from -inf to 0.66434, saving model to vgg16_best.keras
430/430 ━━━━━━━━━━━━━━━━━━━━ 143s 294ms/step - accuracy: 0.4711 - loss: 1.2939 - val_accuracy: 0.7515 - val_loss: 0.6643 - learning_rate: 3.0000e-04
Epoch 2/30
430/430 ━━━━━━━━━━━━━━━━━━━━ 0s 188ms/step - accuracy: 0.7623 - loss: 0.6589
Epoch 2: val_loss did not improve from 0.66434
430/430 ━━━━━━━━━━━━━━━━━━━━ 99s 231ms/step - accuracy: 0.7623 - loss: 0.6588 - val_accuracy: 0.8218 - val_loss: 0.4906 - learning_rate: 3.0000e-04
Epoch 3/30
430/430 ━━━━━━━━━━━━━━━━━━━━ 0s 201ms/step - accuracy: 0.8177 - loss: 0.5101
Epoch 3: val_loss did not improve from 0.66434
430/430 ━━━━━━━━━━━━━━━━━━━━ 105s 245ms/step - accuracy: 0.8177 - loss: 0.5100 - val_accuracy: 0.8489 - val_loss: 0.4143 - learning_rate: 3.0000e-04
Epoch 4/30
430/430 ━━━━━━━━━━━━━━━━━━━━ 0s 202ms/step - accuracy: 0.8372 - loss: 0.4446
Epoch 4: val_loss did not improve fro

PRINT BEST EPOCH BASED ON LOWEST VALIDATION LOSS

In [ ]:
import numpy as np

train_loss = history_vgg16.history['loss']
val_loss   = history_vgg16.history['val_loss']

# Best epoch = minimum validation loss
best_epoch = np.argmin(val_loss) + 1

print("===== TRAINING SUMMARY (BASED ON VALIDATION LOSS) =====")
print(f"Best Epoch              : {best_epoch}")
print(f"Training Loss (Best Ep) : {train_loss[best_epoch-1]:.4f}")
print(f"Validation Loss (Best)  : {val_loss[best_epoch-1]:.4f}")


===== TRAINING SUMMARY (BASED ON VALIDATION LOSS) =====
Best Epoch              : 15
Training Loss (Best Ep) : 0.2357
Validation Loss (Best)  : 0.2084


Also print accuracy at that epoch

In [ ]:
train_acc = history_vgg16.history['accuracy']
val_acc   = history_vgg16.history['val_accuracy']

print(f"Training Accuracy (Best Ep)   : {train_acc[best_epoch-1]*100:.2f}%")
print(f"Validation Accuracy (Best Ep) : {val_acc[best_epoch-1]*100:.2f}%")


Training Accuracy (Best Ep)   : 91.33%
Validation Accuracy (Best Ep) : 92.26%
